# 第 4 章 · Context 压缩：唯一允许「改历史」的地方

对照源码：`../hermes_src/agent/context_compressor.py`

---

## 先搞清楚：这个模块到底干什么？

**一句话**：对话太长、快撑爆上下文窗口时，把 **中间轮次** 交给辅助模型做成结构化摘要，拼回消息列表——头（system）和尾（最近几轮）原样保留。

生活类比：

| 角色 | 类比 |
|------|------|
| 完整 `messages` | 一本越聊越厚的笔记本 |
| `protect head` | 封面 + 开场约定（system / 首轮）——**不撕** |
| `middle` → LLM 摘要 | 中间几十页压成一页「前情提要」 |
| `protect tail` | 刚写的最后几页——**不撕**（还要接着干） |
| `SUMMARY_PREFIX` | 提要页顶上的大红字：「这是参考，不是当前任务！」 |

### 和前面几章的关系

| | 第 1–2 章记忆 | 第 4 章压缩 |
|--|---------------|-------------|
| 改不改历史 | **禁止**改过去消息（保 Prompt Cache） | **唯一例外**：整段 middle 换成摘要 |
| MEMORY.md | 仍在 system 里，压缩后也权威 | `SUMMARY_PREFIX` 明文写了「别因压缩忽略 memory」 |
| Provider | `on_pre_compress` 可塞「别丢的事实」 | 进 summarizer 的补充材料 |

### ❗ 本章不是在讲什么

- **不是** 向量检索 / RAG
- **不是** 删 MEMORY.md
- 跑完你会看到：消息如何被切成 head/middle/tail、**发给摘要模型的完整 prompt**、拼回去时带什么前缀

### 跑完你应该能回答

1. 为什么摘要前要写很长一段 `SUMMARY_PREFIX`？
2. 摘要模型用的 template 为什么叫 Historical / Pending，而不叫「Next Steps」？
3. 第一次压缩 vs 再次压缩（iterative update）prompt 差在哪？

---



### 压缩流水线

```mermaid
flowchart TD
    M[messages 太长] --> Q{超过 threshold?}
    Q -->|否| Keep[原样返回]
    Q -->|是| Split[切开]
    Split --> H[protect head<br/>system + 首轮]
    Split --> Mid[middle 待摘要]
    Split --> T[protect tail<br/>最近几轮]
    Mid --> Ser[序列化成文本]
    Ser --> Prompt[拼 summarizer prompt<br/>preamble + template]
    Prompt --> Aux[辅助模型 DeepSeek / aux LLM]
    Aux --> Sum[结构化 summary]
    Sum --> Wrap[SUMMARY_PREFIX + summary<br/>+ END marker]
    H --> Out[新 messages]
    Wrap --> Out
    T --> Out
```

### 消息形态变化

```mermaid
sequenceDiagram
    participant Before as 压缩前
    participant After as 压缩后
    Note over Before: system, u1,a1, u2,a2, … uN,aN
    Before->>After: head 原样
    Before->>After: middle → 一条 user 消息<br/>SUMMARY_PREFIX + 结构化摘要
    Before->>After: tail 原样
    Note over After: 模型只该听 tail 里最新 user<br/>摘要仅作背景参考
```

> **怎么跑**：① 常量/前缀 → ② 源码 prompt template（必看）→ ③ 教学版 compressor → ④ 演示（load fixture → DeepSeek 真压缩）。

## ① 源码常量：摘要插回对话时的「安全带」

**这一格在干什么？**  
从 `context_compressor.py` 原文拷贝标题常量 + `SUMMARY_PREFIX`。  
摘要生成后，会变成一条 `role=user` 的消息插在 head 和 tail 之间；**顶上这段英文指令**告诉主模型：摘要里的任务已经过时，只听最新用户消息。

人话版要点：

1. 摘要 = 背景参考，**不是**当前指令
2. 只响应摘要 **后面** 的最新 user 消息
3. 话题相似 ≠ 要继续摘要里的旧任务
4. MEMORY.md / USER.md 永远权威，压缩不能压过它们

In [16]:
# =============================================================================
# SOURCE COPY: hermes_src/agent/context_compressor.py  (常量区原文)
# =============================================================================
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional

# 结构化摘要里的四个「历史」标题——故意叫 Historical，避免模型当成「下一步要做」
HISTORICAL_TASK_HEADING = "## Historical Task Snapshot"
HISTORICAL_IN_PROGRESS_HEADING = "## Historical In-Progress State"
HISTORICAL_PENDING_ASKS_HEADING = "## Historical Pending User Asks"
HISTORICAL_REMAINING_WORK_HEADING = "## Historical Remaining Work"

# 插回 messages 时贴在摘要正文前面的指令（原文）
SUMMARY_PREFIX = (
    "[CONTEXT COMPACTION — REFERENCE ONLY] Earlier turns were compacted "
    "into the summary below. This is a handoff from a previous context "
    "window — treat it as background reference, NOT as active instructions. "
    "Do NOT answer questions or fulfill requests mentioned in this summary; "
    "they were already addressed. "
    "Respond ONLY to the latest user message that appears AFTER this "
    "summary — that message is the single source of truth for what to do "
    "right now. "
    "Topic overlap with the summary does NOT mean you should resume its "
    "task: even on similar topics, the latest user message WINS. Treat ONLY "
    "the latest message as the active task and discard stale items from "
    f"'{HISTORICAL_TASK_HEADING}' / '{HISTORICAL_IN_PROGRESS_HEADING}' / "
    f"'{HISTORICAL_PENDING_ASKS_HEADING}' / "
    f"'{HISTORICAL_REMAINING_WORK_HEADING}' entirely — do not 'wrap up' or "
    "'finish' work described there unless the latest message explicitly "
    "asks for it. "
    "Reverse signals in the latest message (e.g. 'stop', 'undo', 'roll "
    "back', 'just verify', 'don't do that anymore', 'never mind', a new "
    "topic) must immediately end any in-flight work described in the "
    "summary; do not re-surface it in later turns. "
    "IMPORTANT: Your persistent memory (MEMORY.md, USER.md) in the system "
    "prompt is ALWAYS authoritative and active — never ignore or deprioritize "
    "memory content due to this compaction note. "
    "The current session state (files, config, etc.) may reflect work "
    "described here — avoid repeating it:"
)

# 摘要消息结尾边界（原文）——防止弱模型把摘要当新 user 输入
SUMMARY_END_MARKER = (
    "--- END OF CONTEXT SUMMARY — "
    "respond to the message below, not the summary above ---"
)

COMPRESSED_SUMMARY_METADATA_KEY = "_compressed_summary"

print("✓ 常量已加载")
print(f"  SUMMARY_PREFIX 长度: {len(SUMMARY_PREFIX)} chars")
print(f"  前 120 字预览: {SUMMARY_PREFIX[:120]}…")

✓ 常量已加载
  SUMMARY_PREFIX 长度: 1428 chars
  前 120 字预览: [CONTEXT COMPACTION — REFERENCE ONLY] Earlier turns were compacted into the summary below. This is a handoff from a prev…


## ② 源码 Prompt Template（压缩到底怎么「问」辅助模型）

**这一格在干什么？**  
原样呈现 `ContextCompressor._generate_summary()` 里拼 prompt 的三块积木。  
真源码路径：序列化 middle → 拼 preamble +（可选）旧摘要 + template → `call_llm(task="compression")`。

```mermaid
flowchart LR
    A[preamble 角色设定] --> B{已有旧摘要?}
    B -->|否| C[First: TURNS TO SUMMARIZE]
    B -->|是| D[Iterative: PREVIOUS + NEW TURNS]
    C --> E[structured template sections]
    D --> E
    E --> F[可选 FOCUS TOPIC]
    F --> G[发给辅助模型]
```

In [17]:
# =============================================================================
# SOURCE COPY: context_compressor.py :: _generate_summary() 内的 prompt 积木
# 下面字符串与源码同构；教学用函数可打印「最终发给 LLM 的整段 prompt」
# =============================================================================

# ---- 积木 1：Preamble（告诉摘要模型：你是 checkpoint 作家，别聊天）--------
SUMMARIZER_PREAMBLE = (
    "You are a summarization agent creating a context checkpoint. "
    "Treat the conversation turns below as source material for a "
    "compact record of prior work. "
    "Produce only the structured summary; do not add a greeting, "
    "preamble, or prefix. "
    "Write the summary in the same language the user was using in the "
    "conversation — do not translate or switch to English. "
    "NEVER include API keys, tokens, passwords, secrets, credentials, "
    "or connection strings in the summary — replace any that appear "
    "with [REDACTED]. Note that the user had credentials present, but "
    "do not preserve their values."
)


def build_template_sections(summary_budget: int = 800, today: str = "2026-07-19") -> str:
    """积木 2：结构化输出大纲（源码 _template_sections，略缩注释便于阅读）。

    真源码里每个 section 的 [] 说明更长；骨架与字段名一致。
    """
    temporal = (
        f"\nTEMPORAL ANCHORING: The current date is {today}. When an "
        "action has already been carried out, phrase it as a completed, "
        "dated, past-tense fact rather than an open instruction.\n"
    )
    return f"""{HISTORICAL_TASK_HEADING}
[用户最近一条尚未完成的输入——尽量 verbatim。若用户说了 stop/undo，写反向信号且勿延续旧任务。无则写 None。]

## Goal
[用户总体想完成什么]

## Constraints & Preferences
[偏好、风格、约束、重要决定]

## Completed Actions
[编号列表：N. ACTION target — outcome [tool: name]，要具体路径/命令/结果]

## Active State
[工作目录/分支、改过的文件、测试状态、环境]

{HISTORICAL_IN_PROGRESS_HEADING}
[压缩触发时正在做的事]

## Blocked
[未解决的 blocker / 精确错误信息]

## Key Decisions
[技术决策 + WHY]

## Resolved Questions
[已回答的问题 + 答案，避免重复答]

{HISTORICAL_PENDING_ASKS_HEADING}
[未完成的用户请求——标成 STALE，除非最新消息明确要求，否则勿执行。无则 None。]

## Relevant Files
[读过/改过/建过的文件 + 简注]

{HISTORICAL_REMAINING_WORK_HEADING}
[剩余工作——仅参考；非最新消息要求勿自动 resume]

## Critical Context
[否则会丢的具体值/报错/配置。密钥写 [REDACTED]]

Target ~{summary_budget} tokens. Be CONCRETE — file paths, outputs, errors, line numbers.
{temporal}
Write only the summary body. Do not include any preamble or prefix."""


def build_summarizer_prompt(
    content_to_summarize: str,
    *,
    previous_summary: str = "",
    summary_budget: int = 800,
    focus_topic: str = "",
    today: str = "2026-07-19",
) -> str:
    """拼出与源码同结构的完整 user prompt（第一次压缩 / 迭代更新两条路径）。"""
    template = build_template_sections(summary_budget, today)

    if previous_summary:
        # 源码 iterative-update 路径
        prompt = f"""{SUMMARIZER_PREAMBLE}

You are updating a context compaction summary. A previous compaction produced the summary below. New conversation turns have occurred since then and need to be incorporated.

PREVIOUS SUMMARY:
{previous_summary}

NEW TURNS TO INCORPORATE:
{content_to_summarize}

Update the summary using this exact structure. PRESERVE all existing information that is still relevant. ADD new completed actions to the numbered list (continue numbering). Move items from "In Progress" to "Completed Actions" when done. Move answered questions to "Resolved Questions". Update "Active State" to reflect current state. Remove information only if it is clearly obsolete. CRITICAL: Update "## Active Task" / Historical Task Snapshot to reflect the user's most recent unfulfilled input.

{template}"""
        path = "iterative-update（已有旧摘要）"
    else:
        # 源码 first-compaction 路径
        prompt = f"""{SUMMARIZER_PREAMBLE}

Create a structured checkpoint summary for the conversation after earlier turns are compacted. The summary should preserve enough detail for continuity without re-reading the original turns.

TURNS TO SUMMARIZE:
{content_to_summarize}

Use this exact structure:

{template}"""
        path = "first-compaction（从零摘要）"

    if focus_topic:
        # 源码 /compress <focus> 追加段
        prompt += f"""

FOCUS TOPIC: "{focus_topic}"
This compaction should PRIORITISE preserving all information related to the focus topic above. For content related to "{focus_topic}", include full detail. For unrelated content, summarise more aggressively. NEVER preserve API keys — use [REDACTED]."""

    print(f"  [build_prompt] 路径 = {path} | prompt 长度 = {len(prompt)} chars")
    return prompt


# 快速看一眼 template 长什么样（不含对话正文）
print("✓ Prompt 积木已定义")
print("\n--- template sections 预览（前 800 字）---\n")
print(build_template_sections())
print("\n…")

✓ Prompt 积木已定义

--- template sections 预览（前 800 字）---

## Historical Task Snapshot
[用户最近一条尚未完成的输入——尽量 verbatim。若用户说了 stop/undo，写反向信号且勿延续旧任务。无则写 None。]

## Goal
[用户总体想完成什么]

## Constraints & Preferences
[偏好、风格、约束、重要决定]

## Completed Actions
[编号列表：N. ACTION target — outcome [tool: name]，要具体路径/命令/结果]

## Active State
[工作目录/分支、改过的文件、测试状态、环境]

## Historical In-Progress State
[压缩触发时正在做的事]

## Blocked
[未解决的 blocker / 精确错误信息]

## Key Decisions
[技术决策 + WHY]

## Resolved Questions
[已回答的问题 + 答案，避免重复答]

## Historical Pending User Asks
[未完成的用户请求——标成 STALE，除非最新消息明确要求，否则勿执行。无则 None。]

## Relevant Files
[读过/改过/建过的文件 + 简注]

## Historical Remaining Work
[剩余工作——仅参考；非最新消息要求勿自动 resume]

## Critical Context
[否则会丢的具体值/报错/配置。密钥写 [REDACTED]]

Target ~800 tokens. Be CONCRETE — file paths, outputs, errors, line numbers.

TEMPORAL ANCHORING: The current date is 2026-07-19. When an action has already been carried out, phrase it as a completed, dated, past-tense fact rather than an open instruction.

Write only the 

## ③ 教学版 `ContextCompressor`（对齐切割 + 拼装骨架）

**这一格在干什么？**  
可跑的压缩器：用固定条数保头保尾（真源码按 **token budget** 保护 tail，还会先 prune tool 输出）。  
演示格会把 `summarize_fn` 接到 DeepSeek（真压缩）；完整 prompt / 摘要落盘到 md。

每个阶段打 `[compress]` 日志，方便边跑边讲。

In [18]:
def _log(tag: str, msg: str) -> None:
    print(f"  [{tag}] {msg}")


def estimate_tokens_rough(messages: List[dict]) -> int:
    """粗估 token：约 chars/4（真源码有更细的 estimate_messages_tokens_rough）。"""
    total = 0
    for m in messages:
        c = m.get("content") or ""
        if isinstance(c, list):
            c = " ".join(p.get("text", "") for p in c if isinstance(p, dict))
        total += len(str(c)) // 4 + 8
    return total


def serialize_for_summary(turns: List[dict]) -> str:
    """把 middle 消息打成「role: content」文本，喂给摘要 prompt。"""
    lines = []
    for m in turns:
        c = m.get("content") or ""
        if isinstance(c, list):
            c = " ".join(p.get("text", "") for p in c if isinstance(p, dict))
        lines.append(f"{m.get('role')}: {c}")
    return "\n".join(lines)


@dataclass
class CompressResult:
    messages: List[dict]
    did_compress: bool
    before_tokens: int
    after_tokens: int
    summary: str = ""
    summarizer_prompt: str = ""  # 教学用：记下发给辅助模型的完整 prompt


class ContextCompressor:
    """教学版：对齐真 ContextCompressor 的切割 + SUMMARY_PREFIX 拼装。

    省略：线程/cooldown、tool prune、token-budget tail、merged-into-tail 奇偶角色修复。
    """

    def __init__(
        self,
        summarize_fn: Callable[[str], str],
        threshold_percent: float = 0.50,
        protect_first_n: int = 1,
        protect_last_n: int = 2,
        context_length: int = 800,  # 教学小窗口，方便触发
    ):
        self.summarize_fn = summarize_fn
        # summarize_fn 实际吃的是「整段 prompt」；下面 compress 会先 build 再传入
        self.threshold_percent = threshold_percent
        self.protect_first_n = protect_first_n
        self.protect_last_n = protect_last_n
        self.threshold_tokens = int(context_length * threshold_percent)
        self._previous_summary = ""  # 迭代压缩时带上
        _log("init", f"threshold_tokens={self.threshold_tokens} "
             f"protect_first={protect_first_n} protect_last={protect_last_n}")

    def compress(
        self,
        messages: List[dict],
        *,
        provider_hint: str = "",
        focus_topic: str = "",
        force: bool = False,
    ) -> CompressResult:
        before = estimate_tokens_rough(messages)
        _log("compress", f"入参 {len(messages)} 条消息 | 粗估 {before} tokens | force={force}")

        if not force and before < self.threshold_tokens:
            _log("compress", f"未超阈值 {self.threshold_tokens} → 不压缩")
            return CompressResult(messages, False, before, before)

        need = self.protect_first_n + self.protect_last_n + 1
        if len(messages) <= need:
            _log("compress", f"消息太少（≤{need}）→ 不压缩")
            return CompressResult(messages, False, before, before)

        head = messages[: self.protect_first_n]
        tail = messages[-self.protect_last_n :]
        middle = messages[self.protect_first_n : -self.protect_last_n]
        _log("compress", f"切开: head={len(head)} | middle={len(middle)} | tail={len(tail)}")
        _log("compress", f"  head roles={[m.get('role') for m in head]}")
        _log("compress", f"  tail roles={[m.get('role') for m in tail]}")

        blob = serialize_for_summary(middle)
        if provider_hint:
            # 对应真链路里 MemoryProvider.on_pre_compress() 的贡献
            blob += f"\n\nExtra facts to preserve (from memory provider):\n{provider_hint}"
            _log("compress", f"附上 provider_hint ({len(provider_hint)} chars)")

        prompt = build_summarizer_prompt(
            blob,
            previous_summary=self._previous_summary,
            focus_topic=focus_topic,
        )
        _log("compress", f"调用 summarize_fn（辅助模型）…")
        summary = self.summarize_fn(prompt)
        self._previous_summary = summary  # 下次走 iterative 路径

        # 插回对话：SUMMARY_PREFIX + 正文 + END marker（对齐源码拼装意图）
        content = f"{SUMMARY_PREFIX}\n\n{summary}\n\n{SUMMARY_END_MARKER}"
        summary_msg = {
            "role": "user",
            "content": content,
            COMPRESSED_SUMMARY_METADATA_KEY: True,  # 前端可识别；上线前 wire 会剥 _* key
        }
        new_messages = head + [summary_msg] + tail
        after = estimate_tokens_rough(new_messages)
        _log("compress", f"拼装完成: {before} → {after} tokens | roles={[m.get('role') for m in new_messages]}")
        return CompressResult(
            new_messages, True, before, after,
            summary=summary, summarizer_prompt=prompt,
        )


print("✓ ContextCompressor 教学版已定义")

✓ ContextCompressor 教学版已定义


## ④ 演示：load 预存对话 → DeepSeek 真压缩 → 按流水线步骤落盘

**故事线**：load fixture → DeepSeek 压缩 → 每个步骤一个目录的 md。

| STEP | 做什么 |
|------|--------|
| 1 | **load** `fixtures/long_conversation.md` |
| 2 | **DeepSeek 真压缩** |
| 3 | 看压缩后形态 |
| 4 | 落盘：`01_切开/`（head/middle/tail）→ `02_middle摘要/`（prompt/摘要）→ `03_拼接/` |


In [ ]:
import json
import os
import re
from pathlib import Path


def _banner(step: str, title: str) -> None:
    print("\n" + "=" * 60)
    print(f"  STEP {step} · {title}")
    print("=" * 60)


def load_messages_md(path: Path) -> list:
    """解析 fixtures/long_conversation.md（===== [i] role=... ===== 块）。"""
    text = path.read_text(encoding="utf-8")
    pattern = re.compile(
        r"^===== \[(\d+)\] role=(\w+)(?:[^\n]*) =====\s*\n",
        re.MULTILINE,
    )
    matches = list(pattern.finditer(text))
    if not matches:
        raise ValueError(f"未在 {path} 中找到消息块")

    messages = []
    for i, m in enumerate(matches):
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].rstrip("\n")
        messages.append({"role": m.group(2), "content": content})
    return messages


# 尝试从仓库常见位置加载 .env（不覆盖已有环境变量）
try:
    from dotenv import load_dotenv

    _here = Path.cwd().resolve()
    for _cand in [
        _here / ".env",
        _here.parent / ".env",
        _here.parent.parent / ".env",
        _here.parent.parent.parent / ".env",
    ]:
        if _cand.is_file():
            load_dotenv(_cand, override=False)
            print(f"(已尝试 load_dotenv: {_cand})")
            break
except ImportError:
    pass


# ---- STEP 1：load 预存多轮对话 ---------------------------------------------
_banner("1", "load fixtures/long_conversation.md")
_FIXTURE = Path("fixtures") / "long_conversation.md"
if not _FIXTURE.is_file():
    raise FileNotFoundError(f"缺少演示对话文件: {_FIXTURE.resolve()}")

messages = load_messages_md(_FIXTURE)
print(f"  已加载 {_FIXTURE}")
print(f"  共 {len(messages)} 条 | 粗估 tokens ≈ {estimate_tokens_rough(messages)}")
print(f"  roles = {[m['role'] for m in messages]}")

# ---- STEP 2：DeepSeek 真压缩 -----------------------------------------------
_banner("2", "DeepSeek 真压缩（需要 DEEPSEEK_API_KEY）")
api_key = (os.getenv("DEEPSEEK_API_KEY") or "").strip()
if not api_key:
    raise RuntimeError(
        "未设置 DEEPSEEK_API_KEY。请在仓库 .env 或环境变量中配置后再跑本格。"
    )

from openai import OpenAI

client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")


def deepseek_summarize(prompt: str) -> str:
    # 真源码用 call_llm(task="compression")；这里等价：整段 prompt 作 user 消息
    _log("deepseek", f"调用 deepseek-chat，prompt={len(prompt)} chars")
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )
    return (resp.choices[0].message.content or "").strip()


comp = ContextCompressor(
    summarize_fn=deepseek_summarize,
    protect_first_n=3,
    protect_last_n=2,
    context_length=400,
)
result = comp.compress(
    messages,
    provider_hint="用户叫 Julie；目标方向 AI Engineer（来自 on_pre_compress）",
    force=True,
)

print(f"  压缩: {result.did_compress} | tokens {result.before_tokens} → {result.after_tokens}")
print(f"  roles: {[m['role'] for m in result.messages]}")
print(f"  summarizer_prompt: {len(result.summarizer_prompt)} chars（见 STEP 4 落盘 md）")
print(f"  摘要正文: {len(result.summary)} chars（见 STEP 4 落盘 md）")

# ---- STEP 3：看形态 --------------------------------------------------------
_banner("3", "压缩后消息形态")
roles = [m["role"] for m in result.messages]
mid = result.messages[1]
print(f"  head[0]={roles[0]} | mid[1]=摘要消息 | tail={roles[-2:]}")
print(f"  mid 带 SUMMARY_PREFIX: {SUMMARY_PREFIX[:40] in mid['content']}")
print(f"  mid 带 {COMPRESSED_SUMMARY_METADATA_KEY}: {bool(mid.get(COMPRESSED_SUMMARY_METADATA_KEY))}")
print("  下一格 STEP 4 落盘: 01_切开/ → 02_middle摘要/ → 03_拼接/")


(已尝试 load_dotenv: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\.env)

  STEP 1 · load fixtures/long_conversation.md
  已加载 fixtures\long_conversation.md
  共 17 条 | 粗估 tokens ≈ 922
  roles = ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']

  STEP 2 · DeepSeek 真压缩（需要 DEEPSEEK_API_KEY）
  [init] threshold_tokens=200 protect_first=1 protect_last=2
  [compress] 入参 17 条消息 | 粗估 922 tokens | force=True
  [compress] 切开: head=1 | middle=14 | tail=2
  [compress]   head roles=['system']
  [compress]   tail roles=['user', 'assistant']
  [compress] 附上 provider_hint (46 chars)
  [build_prompt] 路径 = first-compaction（从零摘要） | prompt 长度 = 4609 chars
  [compress] 调用 summarize_fn（辅助模型）…
  [deepseek] 调用 deepseek-chat，prompt=4609 chars


In [ ]:
import shutil

# ---- STEP 4：按「压缩流水线步骤」分目录落盘 ----
_banner("4", "写入 exports/context_compression/{切开,middle摘要,拼接}/")

_EXPORT = Path("exports") / "context_compression"
# 清空旧产物，只保留本流水线目录结构
if _EXPORT.exists():
    shutil.rmtree(_EXPORT)
_EXPORT.mkdir(parents=True, exist_ok=True)

_DIR_SPLIT = _EXPORT / "01_切开"
_DIR_SUM = _EXPORT / "02_middle摘要"
_DIR_JOIN = _EXPORT / "03_拼接"
for _d in (_DIR_SPLIT, _DIR_SUM, _DIR_JOIN):
    _d.mkdir(parents=True, exist_ok=True)

# 与 compress() 一致
_PROTECT_FIRST_N = 3
_PROTECT_LAST_N = 2
_head = messages[:_PROTECT_FIRST_N]
_tail = messages[-_PROTECT_LAST_N:]
_middle = messages[_PROTECT_FIRST_N : -_PROTECT_LAST_N]
_summary_body = result.summary
_wrapped = result.messages[1]["content"]
_assembled = result.messages


def _write(path: Path, text: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text if text.endswith("\n") else text + "\n", encoding="utf-8")
    print(f"  → {path.relative_to(_EXPORT)}  ({len(text)} chars)")
    return path


def _fmt_messages(msgs: list, *, start_index: int = 0) -> str:
    parts = []
    for i, m in enumerate(msgs):
        role = m.get("role", "?")
        content = m.get("content") or ""
        if isinstance(content, list):
            content = " ".join(
                p.get("text", "") for p in content if isinstance(p, dict)
            )
        meta = ""
        if m.get(COMPRESSED_SUMMARY_METADATA_KEY):
            meta = "  [_compressed_summary=True]"
        parts.append(f"===== [{start_index + i}] role={role}{meta} =====\n{content}\n")
    return "\n".join(parts)


# ========== 步骤 1：切开 Split → head / middle / tail ==========
_sys = (
    messages[0]["content"]
    if messages and messages[0].get("role") == "system"
    else ""
)
_write(
    _DIR_SPLIT / "README.md",
    "# 步骤 1 · 切开（Split）\n\n"
    "对应逻辑图：`messages → Split → head / middle / tail`\n\n"
    f"- protect_first_n = {_PROTECT_FIRST_N}\n"
    f"- protect_last_n = {_PROTECT_LAST_N}\n"
    f"- head={len(_head)} | middle={len(_middle)} | tail={len(_tail)}\n\n"
    "| 文件 | 含义 |\n"
    "|------|------|\n"
    "| `system_prompt.md` | head 里的 system / MEMORY |\n"
    "| `head.md` | protect head（原样保留） |\n"
    "| `middle.md` | 待摘要的中间轮次 |\n"
    "| `tail.md` | protect tail（原样保留） |\n",
)
_write(
    _DIR_SPLIT / "system_prompt.md",
    "# system prompt\n\n"
    "> 属于 **head**；不进 middle，不被摘要改写。\n\n"
    f"{_sys}\n",
)
_write(
    _DIR_SPLIT / "head.md",
    "# head（protect head）\n\n"
    f"- 条数: {len(_head)}\n"
    f"- roles: {[m.get('role') for m in _head]}\n\n"
    + _fmt_messages(_head, start_index=0),
)
_write(
    _DIR_SPLIT / "middle.md",
    "# middle（待摘要）\n\n"
    f"- 条数: {len(_middle)}\n"
    f"- roles: {[m.get('role') for m in _middle]}\n"
    f"- 粗估 tokens: {estimate_tokens_rough(_middle)}\n\n"
    + _fmt_messages(_middle, start_index=_PROTECT_FIRST_N),
)
_write(
    _DIR_SPLIT / "tail.md",
    "# tail（protect tail）\n\n"
    f"- 条数: {len(_tail)}\n"
    f"- roles: {[m.get('role') for m in _tail]}\n\n"
    + _fmt_messages(_tail, start_index=len(messages) - _PROTECT_LAST_N),
)

# ========== 步骤 2：middle → 压缩 prompt → 摘要 ==========
_write(
    _DIR_SUM / "README.md",
    "# 步骤 2 · middle 摘要\n\n"
    "对应逻辑图：`middle → 序列化 → 压缩 prompt → Aux LLM → summary → Wrap`\n\n"
    "| 文件 | 含义 |\n"
    "|------|------|\n"
    "| `压缩prompt.md` | 发给辅助模型的完整 prompt |\n"
    "| `摘要.md` | summarize_fn 纯输出（结构化摘要正文） |\n"
    "| `包装后的摘要消息.md` | SUMMARY_PREFIX + 摘要 + END（插回用） |\n",
)
_write(
    _DIR_SUM / "压缩prompt.md",
    "# middle · 压缩 prompt\n\n"
    "> = PREAMBLE + TURNS TO SUMMARIZE(middle) + Template\n\n"
    f"- 长度: {len(result.summarizer_prompt)} chars\n\n"
    "---\n\n"
    f"{result.summarizer_prompt}\n",
)
_write(
    _DIR_SUM / "摘要.md",
    "# middle · 摘要（压缩完成的内容）\n\n"
    "> 辅助模型对 **middle** 的结构化摘要正文（尚未包 SUMMARY_PREFIX）。\n\n"
    f"{_summary_body}\n",
)
_write(
    _DIR_SUM / "包装后的摘要消息.md",
    "# middle · 包装后的摘要消息（Wrap）\n\n"
    "> 对应逻辑图：`summary → Wrap(SUMMARY_PREFIX + summary + END)`\n"
    "> 插回 messages 时 role=`user`，并带 `_compressed_summary`。\n\n"
    f"{_wrapped}\n",
)

# ========== 步骤 3：拼接 Out = head + 摘要消息 + tail ==========
_write(
    _DIR_JOIN / "README.md",
    "# 步骤 3 · 拼接\n\n"
    "对应逻辑图：`Out = head + Wrap(摘要消息) + tail`\n\n"
    "| 文件 | 含义 |\n"
    "|------|------|\n"
    "| `拼接的context.md` | 压缩后发给主模型的完整 messages |\n"
    "| `拼接的context.json` | 同上（机器可读） |\n",
)
_write(
    _DIR_JOIN / "拼接的context.md",
    "# 拼接的 context\n\n"
    f"- 条数: {len(_assembled)}\n"
    f"- 粗估 tokens: {estimate_tokens_rough(_assembled)}\n"
    f"- roles: {[m.get('role') for m in _assembled]}\n"
    f"- 公式: head({len(_head)}) + summary_msg(1) + tail({len(_tail)})\n\n"
    + _fmt_messages(_assembled),
)
_write(
    _DIR_JOIN / "拼接的context.json",
    json.dumps(_assembled, ensure_ascii=False, indent=2),
)

# 根目录总览
_write(
    _EXPORT / "README.md",
    """# Context Compression 流水线产物

```text
01_切开/          ← Split：system / head / middle / tail
02_middle摘要/    ← middle → 压缩prompt → 摘要 → 包装消息
03_拼接/          ← head + 摘要消息 + tail
```

讲解顺序：`01_切开` → `02_middle摘要` → `03_拼接`。
""",
)

print(f"\n[ok] 流水线目录: {_EXPORT.resolve()}")
print(
    f"  01_切开: head={len(_head)} middle={len(_middle)} tail={len(_tail)}"
)
print(f"  02_middle摘要: prompt={len(result.summarizer_prompt)} chars, summary={len(_summary_body)} chars")
print(f"  03_拼接: {len(_assembled)} messages")


## 跑完对照：源码压缩到底做了什么？

```text
messages = [system, u1,a1, u2,a2, … uN,aN]
                 │
                 ├─ head  : system（+ 可选首轮）     ← 不改（Cache / 人设）
                 ├─ middle: 序列化 → 拼 prompt：
                 │            SUMMARIZER_PREAMBLE
                 │          + TURNS TO SUMMARIZE  (或 PREVIOUS + NEW)
                 │          + 结构化 template（Historical* 等 section）
                 │          + 可选 FOCUS TOPIC
                 │         → 辅助模型 → summary 正文
                 │         → 包上 SUMMARY_PREFIX + END marker
                 └─ tail  : 最近几轮                 ← 不改（当前任务）
```

**设计意图（面试可说）**：

1. **唯一破 Cache 的合法点**：只有压缩会改历史；日常记忆走 snapshot / prefetch。
2. **双保险防「摘要劫持」**：`SUMMARY_PREFIX`（给主模型）+ Historical 标题（给摘要模型）——旧任务标成 STALE。
3. **迭代压缩**：第二次起带上 `PREVIOUS SUMMARY`，避免多压几次把关键事实磨没。
4. **Memory 仍权威**：前缀明文要求勿因 compaction 忽略 MEMORY.md / USER.md。

**真源码比本 notebook 多的部分**（知道即可）：按 token 保护 tail、先 prune 大 tool 输出、摘要失败 cooldown、奇偶角色时把摘要 merge 进 tail、`on_pre_compress` 经 MemoryManager 汇入。